In [0]:
%pip install lxml==6.1.3 html5lib==1.1

In [0]:
dbutils.library.restartPython()

In [0]:
%run ../../_shared/00_project_config

In [0]:
import re
from pathlib import Path

import pandas as pd
from pyspark.sql import functions as F


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
# Project-wide paths, schemas, supported leagues, seasons, and FBref datasets
# are loaded from 00_project_config.
#
# FBREF_LEAGUES is a mapping:
#   source league code -> canonical platform league name
#
# The Bronze table naming convention intentionally continues to use the
# source league code because that matches the existing project structure.

SOURCE_SYSTEM = "fbref"


# -----------------------------------------------------------------------------
# Helper: flatten FBref column headers
# -----------------------------------------------------------------------------
# FBref tables frequently contain multi-level HTML headers. pandas.read_html()
# represents those headers as tuples / MultiIndex values.
#
# Bronze keeps the source-oriented column names, but they must first be
# flattened into a single level so Spark and Delta can store them.
#
# Only characters that are problematic for Delta column names are replaced.
# Other source characters such as %, #, /, + and - are intentionally retained
# because existing Silver transformations already reference fields such as:
#   Standard_SoT%
#   Starts_Mn/Start
#   Team_Success_+/-
#   #_Pl

def flatten_columns(columns) -> list[str]:
    """Flatten FBref multi-level headers into Delta-compatible column names."""

    flattened_columns = []

    for column in columns:
        if isinstance(column, tuple):
            parts = []

            for level in column:
                level_name = str(level).strip()

                if (
                    level_name
                    and level_name.lower() != "nan"
                    and not level_name.startswith("Unnamed:")
                ):
                    parts.append(level_name)

            column_name = "_".join(parts)

        else:
            column_name = str(column).strip()

        # Replace characters that are problematic in Delta column names.
        column_name = re.sub(
            r"[ ,;{}()\n\t=]",
            "_",
            column_name,
        )

        if not column_name:
            raise ValueError(
                "FBref column flattening produced an empty column name."
            )

        flattened_columns.append(column_name)

    return flattened_columns


# -----------------------------------------------------------------------------
# Helper: validate flattened column names
# -----------------------------------------------------------------------------
# Flattening a multi-level header can theoretically cause two different source
# columns to collapse into the same final name. Duplicate Spark column names
# would make downstream transformations ambiguous, so fail early instead of
# silently loading an unsafe schema.

def validate_unique_columns(column_names: list[str]) -> None:
    """Fail if flattened FBref headers contain duplicate column names."""

    seen = set()
    duplicates = set()

    for column_name in column_names:
        if column_name in seen:
            duplicates.add(column_name)

        seen.add(column_name)

    if duplicates:
        raise ValueError(
            "Duplicate FBref column names detected after header flattening: "
            f"{sorted(duplicates)}"
        )


# -----------------------------------------------------------------------------
# Load FBref files into Bronze
# -----------------------------------------------------------------------------
# Each raw file represents one:
#
#   league + season + dataset
#
# combination.
#
# Example:
#   pl_2425_standard_stats.xls
#
# Although the files use an .xls extension, the current ingestion process
# stores HTML table content that is parsed using pandas.read_html().
#
# Bronze responsibilities here are intentionally limited to:
#   - parsing the raw HTML table
#   - flattening the source header hierarchy
#   - adding source context
#   - adding technical lineage metadata
#   - appending the result to Delta
#
# Cleaning, casting, deduplication, business validation, and canonical league
# naming remain Silver responsibilities.

loaded_file_count = 0

for season in FBREF_SEASONS:
    for league_code in FBREF_LEAGUES:
        for dataset in FBREF_DATASETS:

            file_path = (
                f"{FBREF_RAW_ROOT}/"
                f"{league_code}_{season}_{dataset}.xls"
            )

            bronze_table = (
                f"{BRONZE_SCHEMA}."
                f"fbref_{dataset}_{league_code}"
            )

            # -----------------------------------------------------------------
            # Validate raw source file
            # -----------------------------------------------------------------

            if not Path(file_path).exists():
                raise FileNotFoundError(
                    f"Expected FBref source file does not exist: {file_path}"
                )

            # -----------------------------------------------------------------
            # Parse raw FBref HTML table
            # -----------------------------------------------------------------
            # Fail with additional source context if pandas cannot parse the
            # expected table.
            try:
                tables = pd.read_html(
                    file_path,
                    flavor="lxml",
                )

            except Exception as exc:
                raise RuntimeError(
                    "Failed to parse FBref source file.\n"
                    f"File: {file_path}\n"
                    f"Error type: {type(exc).__name__}\n"
                    f"Error: {exc}"
                ) from exc

            if not tables:
                raise ValueError(
                    f"No HTML tables found in FBref source file: {file_path}"
                )

            # Each project source file is expected to contain the target FBref
            # dataset as the first table.
            pdf = tables[0].copy()

            # -----------------------------------------------------------------
            # Flatten and validate source columns
            # -----------------------------------------------------------------

            flattened_columns = flatten_columns(pdf.columns)

            validate_unique_columns(flattened_columns)

            pdf.columns = flattened_columns

            # -----------------------------------------------------------------
            # Convert pandas DataFrame to Spark
            # -----------------------------------------------------------------

            source_df = spark.createDataFrame(pdf)

            # -----------------------------------------------------------------
            # Add Bronze metadata
            # -----------------------------------------------------------------
            # _season and _league retain source identifiers at Bronze level.
            #
            # _source_file is a literal here rather than _metadata.file_path
            # because pandas, not Spark's file reader, reads the raw file.

            bronze_df = (
                source_df
                .withColumn(
                    "_season",
                    F.lit(season),
                )
                .withColumn(
                    "_league",
                    F.lit(league_code),
                )
                .withColumn(
                    "_stat",
                    F.lit(dataset),
                )
                .withColumn(
                    "_ingested_at",
                    F.current_timestamp(),
                )
                .withColumn(
                    "_source_file",
                    F.lit(file_path),
                )
                .withColumn(
                    "_source_system",
                    F.lit(SOURCE_SYSTEM),
                )
            )

            # -----------------------------------------------------------------
            # Write to Bronze Delta table
            # -----------------------------------------------------------------
            # Append mode preserves ingestion history.
            #
            # mergeSchema is intentionally enabled in Bronze because FBref may
            # add source columns over time. Silver remains responsible for
            # enforcing the analytical schema expected by downstream models.

            (
                bronze_df.write
                .format("delta")
                .mode("append")
                .option("mergeSchema", "true")
                .saveAsTable(bronze_table)
            )

            loaded_file_count += 1

            print(
                f"Loaded FBref Bronze file "
                f"{loaded_file_count}: "
                f"{league_code} / {season} / {dataset} "
                f"-> {bronze_table}"
            )


# -----------------------------------------------------------------------------
# Completion message
# -----------------------------------------------------------------------------

print(
    f"Successfully loaded {loaded_file_count} "
    "FBref source files into Bronze."
)